### Import Libraries

In [4]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------------------------------------- 0.0/12.8 MB 640.0 kB/s eta 0:00:20
     --------------------------------------- 0.1/12.8 MB 777.7 kB/s eta 0:00:17
      --------------------------------------- 0.2/12.8 MB 1.2 MB/s eta 0:00:11
      --------------------------------------- 0.3/12.8 MB 1.3 MB/s eta 0:00:10
     - -------------------------------------- 0.4/12.8 MB 1.6 MB/s eta 0:00:08
     - -------------------------------------- 0.5/12.8 MB 1.6 MB/s eta 0:00:08
     - -------------------------------------- 0.6/12.8 MB 1.8 MB/s eta 0:00:07
     -- ------------------------------------- 0.7/12.8 MB 1.8 MB/s eta 0:00:07
     -- ------------------------------------- 0.7/12.8 MB 1.8 MB/s eta 0:00:07
     -- ------------------------------------- 0.9/12.8 MB 1.9 MB/s eta 0:00:07
     -- ------------------------------------- 1.0/12.8 MB 1.9 MB/s eta 0:00:07
     --- ------------------------------------ 1.0/12.8 MB


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
import re
import pandas as pd
import numpy as np
import nltk
import spacy

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

### Download NLTK resources

In [6]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Load spaCy model

In [9]:
# Load spaCy small English model (no trailing space in model name)
nlp = spacy.load("en_core_web_sm")

### Create folder structure for data

In [10]:
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/cleaned", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

### Download Dataset

In [12]:
from pathlib import Path
from urllib.request import urlretrieve

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

train_url = "https://huggingface.co/datasets/stanfordnlp/sentiment140/resolve/refs%2Fconvert%2Fparquet/sentiment140/train/0000.parquet"
test_url = "https://huggingface.co/datasets/stanfordnlp/sentiment140/resolve/refs%2Fconvert%2Fparquet/sentiment140/test/0000.parquet"

train_path = raw_dir / "train.parquet"
test_path = raw_dir / "test.parquet"

urlretrieve(train_url, train_path)
urlretrieve(test_url, test_path)

print(f"Saved: {train_path}")
print(f"Saved: {test_path}")

Saved: data\raw\train.parquet
Saved: data\raw\test.parquet


### Load the datasets

In [17]:
train_df = pd.read_parquet(r"E:\MLP\Algorithms\supervised\NLP_pipeline\data\raw\train.parquet")
test_df = pd.read_parquet(r"E:\MLP\Algorithms\supervised\NLP_pipeline\data\raw\test.parquet") 
train_df.head()  

,text,date,user,sentiment,query
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",Mon Apr 06 22:19:45 PDT 2009,_TheSpecialOne_,0,NO_QUERY
1,is upset that he can't update his Facebook by ...,Mon Apr 06 22:19:49 PDT 2009,scotthamilton,0,NO_QUERY
2,@Kenichan I dived many times for the ball. Man...,Mon Apr 06 22:19:53 PDT 2009,mattycus,0,NO_QUERY
3,my whole body feels itchy and like its on fire,Mon Apr 06 22:19:57 PDT 2009,ElleCTF,0,NO_QUERY
4,"@nationwideclass no, it's not behaving at all....",Mon Apr 06 22:19:57 PDT 2009,Karoli,0,NO_QUERY


### Data Understanding

In [18]:
print("Shape:", train_df.shape)
print("Columns:", train_df.columns)
train_df.head(5)

Shape: (1600000, 5)
Columns: Index(['text', 'date', 'user', 'sentiment', 'query'], dtype='str')


,text,date,user,sentiment,query
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",Mon Apr 06 22:19:45 PDT 2009,_TheSpecialOne_,0,NO_QUERY
1,is upset that he can't update his Facebook by ...,Mon Apr 06 22:19:49 PDT 2009,scotthamilton,0,NO_QUERY
2,@Kenichan I dived many times for the ball. Man...,Mon Apr 06 22:19:53 PDT 2009,mattycus,0,NO_QUERY
3,my whole body feels itchy and like its on fire,Mon Apr 06 22:19:57 PDT 2009,ElleCTF,0,NO_QUERY
4,"@nationwideclass no, it's not behaving at all....",Mon Apr 06 22:19:57 PDT 2009,Karoli,0,NO_QUERY


### Select Required Columns

In [20]:
text_data = train_df[["text", "sentiment"]].dropna()
text_data.head()

,text,sentiment
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0
1,is upset that he can't update his Facebook by ...,0
2,@Kenichan I dived many times for the ball. Man...,0
3,my whole body feels itchy and like its on fire,0
4,"@nationwideclass no, it's not behaving at all....",0


### Check Sentiment distribution

In [21]:
text_data['sentiment'].value_counts()

sentiment
0    800000
4    800000
Name: count, dtype: int64

### Convert sentiment to binary

In [53]:
# Robust binary sentiment conversion: supports 0/4, 0/1, and text labels
sentiment_map = {
    0: 0, 4: 1, 1: 1,
    '0': 0, '4': 1, '1': 1,
    'negative': 0, 'positive': 1
}

text_data['sentiment'] = (
    text_data['sentiment']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(sentiment_map)
)

text_data['sentiment'] = pd.to_numeric(text_data['sentiment'], errors='coerce')
text_data = text_data.dropna(subset=['sentiment']).copy()
text_data['sentiment'] = text_data['sentiment'].astype(int)

print(text_data['sentiment'].value_counts(dropna=False))
text_data.head()

sentiment
0    800000
Name: count, dtype: int64


,text,sentiment,cleaned_text,preprocessed_text
800000,I LOVE @Health4UandPets u guys r the best!!,0,i love u guys r the best,love u guy r best
800001,im meeting up with one of my besties tonight! ...,0,im meeting up with one of my besties tonight c...,im meeting one besties tonight cant wait girl ...
800002,"@DaRealSunisaKim Thanks for the Twitter add, S...",0,thanks for the twitter add sunisa i got to mee...,thanks twitter add sunisa got meet hin show dc...
800003,Being sick can be really cheap when it hurts t...,0,being sick can be really cheap when it hurts t...,sick really cheap hurt much eat real food plus...
800004,@LovesBrooklyn2 he has that effect on everyone,0,he has that effect on everyone,effect everyone


### Text Cleaning function 

In [54]:

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

text_data['cleaned_text'] = text_data['text'].apply(clean_text)
text_data.head()

,text,sentiment,cleaned_text,preprocessed_text
800000,I LOVE @Health4UandPets u guys r the best!!,0,i love u guys r the best,love u guy r best
800001,im meeting up with one of my besties tonight! ...,0,im meeting up with one of my besties tonight c...,im meeting one besties tonight cant wait girl ...
800002,"@DaRealSunisaKim Thanks for the Twitter add, S...",0,thanks for the twitter add sunisa i got to mee...,thanks twitter add sunisa got meet hin show dc...
800003,Being sick can be really cheap when it hurts t...,0,being sick can be really cheap when it hurts t...,sick really cheap hurt much eat real food plus...
800004,@LovesBrooklyn2 he has that effect on everyone,0,he has that effect on everyone,effect everyone


### Saved Cleaned Data

In [55]:
text_data.to_csv("data/cleaned/cleaned_text.csv", index=False)

### Tokenization

In [57]:
sample = text_data['cleaned_text'].iloc[0]
sentences = sent_tokenize(sample)
word = word_tokenize(sample)
print("sentences:", sentences)
print("words:", word)


sentences: ['i love u guys r the best']
words: ['i', 'love', 'u', 'guys', 'r', 'the', 'best']


### Stopword removal

In [58]:
stop_words = set(stopwords.words('english'))

filtered_words = [w for w in word if w not in stop_words]
filtered_words  

['love', 'u', 'guys', 'r', 'best']

### Lemmatization

In [59]:
lemmatizer = WordNetLemmatizer()
lemmatized_words = [lemmatizer.lemmatize(w) for w in filtered_words]
lemmatized_words

['love', 'u', 'guy', 'r', 'best']

### full preprocessing pipeline

In [60]:
def prepocess_pipeline(text):
    text = clean_text(text)
    words = word_tokenize(text)
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(words)
text_data['preprocessed_text'] = text_data['cleaned_text'].apply(prepocess_pipeline)
text_data.head()

,text,sentiment,cleaned_text,preprocessed_text
800000,I LOVE @Health4UandPets u guys r the best!!,0,i love u guys r the best,love u guy r best
800001,im meeting up with one of my besties tonight! ...,0,im meeting up with one of my besties tonight c...,im meeting one besties tonight cant wait girl ...
800002,"@DaRealSunisaKim Thanks for the Twitter add, S...",0,thanks for the twitter add sunisa i got to mee...,thanks twitter add sunisa got meet hin show dc...
800003,Being sick can be really cheap when it hurts t...,0,being sick can be really cheap when it hurts t...,sick really cheap hurt much eat real food plus...
800004,@LovesBrooklyn2 he has that effect on everyone,0,he has that effect on everyone,effect everyone


### Saved processed data

In [61]:
text_data.to_csv("data/processed/processed_text.csv", index=False)

### Named entity recognation

In [62]:
sample_text = text_data['preprocessed_text'].iloc[0]
doc = nlp(sample_text)

for ent in doc.ents:
    print(ent.text, ent.label_)

### POS Tagging

In [63]:
for tocken in doc:
    print(tocken.text, tocken.pos_)


love VERB
u PRON
guy NOUN
r NOUN
best ADJ


### Chunking

In [64]:
for chunk in doc.noun_chunks:
    print(chunk.text)

u
guy


### Dependancy Parsing

In [65]:
for token in doc:
    print(token.text, token.dep_, token.head.text)

love ROOT love
u dobj love
guy dobj love
r advmod best
best advmod love


### Tf-IDF vectorization

In [66]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

# Keep only rows with valid label/text for modeling
model_data = text_data.dropna(subset=['preprocessed_text', 'sentiment']).copy()
model_data['sentiment'] = model_data['sentiment'].astype(int)

X = vectorizer.fit_transform(model_data['preprocessed_text'])
y = model_data['sentiment']

### Train test split

In [67]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train class counts:')
print(y_train.value_counts())
print('Test class counts:')
print(y_test.value_counts())

Train class counts:
sentiment
0    640000
Name: count, dtype: int64
Test class counts:
sentiment
0    160000
Name: count, dtype: int64


### Train model (Logistic regression)

In [ ]:
from sklearn.linear_model import LogisticRegression

print('Classes in y_train:', sorted(y_train.unique().tolist()))

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)